In [ ]:
import glob
import cv2
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import transforms, models
import matplotlib.pyplot as plt
import numpy as np
from torch import nn, optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns
import datetime
import segmentation_models_pytorch as smp
import time
from IPython.display     import display, clear_output

## Load Model

In [ ]:
# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
lung_model_path = '../2026 WyvernGUI/models/lung_MAnet_20260209_042722_best.pth'
rib_model_path = '../2026 WyvernGUI/models/rib_MAnet_20260209_034825_best.pth'
classification_model_path = '../2026 WyvernGUI/models/Classification_resnet34_20260209_051509_best.pth'

In [ ]:
class MAnetWithDropout(nn.Module):
    """MAnet wrapper that applies spatial dropout on decoder features."""

    def __init__(self, dropout_p=0.2, **manet_kwargs):
        super().__init__()
        self.model = smp.MAnet(**manet_kwargs)
        self.dropout = nn.Dropout2d(p=dropout_p)
        self.dropout_p = dropout_p
        
    def name(self):
        return f"MAnetWithDropout_{self.dropout_p}"

    def forward(self, x):
        features = self.model.encoder(x)
        decoder_output = self.model.decoder(*features)
        decoder_output = self.dropout(decoder_output)
        masks = self.model.segmentation_head(decoder_output)

        if self.model.classification_head is not None:
            labels = self.model.classification_head(features[-1])
            return masks, labels

        return masks

In [ ]:
##########################################################################################################################
################################################## Load Rib & Lung model #################################################
##########################################################################################################################

SIZE_X = 512 
SIZE_Y = 512
ENCODER = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
activation = None

print("Loading rib model...")

rib_model = MAnetWithDropout(
                            dropout_p=0.5,
                            encoder_name=ENCODER,
                            encoder_weights=ENCODER_WEIGHTS,
                            in_channels=3,
                            classes=3
                        )
rib_model.to(device)
rib_pre_dict = torch.load(rib_model_path, map_location=device)
rib_model.load_state_dict(rib_pre_dict['model'])
rib_model.eval()
print("✓ Rib model loaded")

print("\nLoading lung model...")
lung_model = MAnetWithDropout(
                            dropout_p=0.5,
                            encoder_name=ENCODER,
                            encoder_weights=ENCODER_WEIGHTS,
                            in_channels=3,
                            classes=3
                        )
lung_model.to(device)
lung_pre_dict = torch.load(lung_model_path, map_location=device)
lung_model.load_state_dict(lung_pre_dict['model'])
lung_model.eval()
print("✓ Lung model loaded")

# Load Classification Model
print("Loading classification model...")
model = models.resnet34(weights='IMAGENET1K_V1').to(device)

model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(model.fc.in_features, 2)
    )
model = model.to(device)

state_dict = torch.load(classification_model_path, map_location=device)
model.load_state_dict(state_dict['model'])
model.eval()
print("✓ Classification model loaded")

print("\n" + "="*60)
print("ALL MODELS LOADED SUCCESSFULLY")
print("="*60)

## Helping Function

In [ ]:
def np_to_torch(np_array):      return torch.from_numpy(np_array).float()

def torch_to_np(torch_array):   return np.squeeze(torch_array.detach().cpu().numpy())

def load_image(image_path):
    """Load and preprocess an image."""
    if image_path.endswith('.dcm'):
        import pydicom
        dicom_data = pydicom.dcmread(image_path)
        img = dicom_data.pixel_array
    else:
        img = cv2.imread(image_path, 1) #Read in BGR mode (1)
    img = cv2.resize(img, (512,512), interpolation = cv2.INTER_NEAREST)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img/255    
    img = img.astype(np.float64)
    image_float = img.copy()
    img = np.transpose(img, (2, 0, 1))
    img = np_to_torch(img)
    return img, image_float

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from PIL import Image

def load_and_show_gradcam(image_path, model, device, target_layer=None, target_class=None, 
                          use_cutoff=True, cutoff_threshold=0.3, show_colorbar=False):
    """
    Load a PNG image, preprocess it, run GradCAM, and display the results.
    
    Args:
        image_path: Path to the PNG image
        model: PyTorch model
        device: torch device (cpu or cuda)
        target_layer: The layer to generate GradCAM for (default: model.layer4[-1])
        target_class: Class index to visualize (default: predicted class)
        use_cutoff: Whether to apply cutoff threshold (default: True)
        cutoff_threshold: Cutoff value for GradCAM (0-1), values below this are set to 0 (default: 0.3)
        show_colorbar: Whether to show colorbar scale (default: True)
    
    Returns:
        visualization: GradCAM visualization overlaid on original image
    """
    # Load image using the same approach as ImageDataset (with augmentation=None, preprocessing=None)
    img, image_float = load_image(image_path)
    input_tensor = img.unsqueeze(0).to(device)
    class_map = {"Adequate inspiration": 0, "Inadequate inspiration": 1}
    
    # Get prediction
    with torch.no_grad():
        output = model(input_tensor)
        print(f"Model output logits: {output}")
        predicted_class = torch.argmax(output, dim=1).item()
        confidence = torch.softmax(output, dim=1)[0, predicted_class].item()
    
    print(f"Predicted class: {list(class_map.keys())[list(class_map.values()).index(predicted_class)]} with confidence: {confidence:.4f}")
    
    # Set target layer (default to last layer of layer4)
    if target_layer is None:
        target_layer = model.layer4[-1]
    
    # Set target class (default to predicted class)
    if target_class is None:
        target_class = predicted_class
    
    # Initialize GradCAM
    cam = GradCAM(model=model, target_layers=[target_layer])
    
    # Generate GradCAM
    grayscale_cam = cam(input_tensor=input_tensor, targets=None if target_class == predicted_class else [target_class])
    grayscale_cam = grayscale_cam[0, :]
    
    # Store original cam for heatmap display
    grayscale_cam_display = grayscale_cam.copy()
    
    # Apply cutoff if enabled
    if use_cutoff:
        print(f"Applying cutoff threshold: {cutoff_threshold} (min: {grayscale_cam.min():.3f}, max: {grayscale_cam.max():.3f})")
        # Create mask for values below threshold
        mask = grayscale_cam < cutoff_threshold
        grayscale_cam = np.where(mask, 0, grayscale_cam)
        # Renormalize after cutoff (only non-zero values)
        if grayscale_cam.max() > 0:
            grayscale_cam = (grayscale_cam - grayscale_cam.min()) / (grayscale_cam.max() - grayscale_cam.min())
        grayscale_cam_display = grayscale_cam.copy()
    
    # Create visualization with transparency for cutoff regions
    if use_cutoff:
        # Create custom overlay with transparency
        from matplotlib import cm
        colormap = cm.get_cmap('jet')
        heatmap = colormap(grayscale_cam)[:, :, :3]  # RGB only
        
        # Create alpha channel based on grayscale_cam (0 where cam is 0, 1 where cam > 0)
        alpha = (grayscale_cam > 0).astype(float) * 0.5  # 0.5 opacity for overlay
        
        # Blend: show original image where alpha is 0, overlay where alpha > 0
        visualization = image_float.copy()
        for i in range(3):  # RGB channels
            visualization[:, :, i] = image_float[:, :, i] * (1 - alpha) + heatmap[:, :, i] * alpha
        visualization = (visualization * 255).astype(np.uint8)
    else:
        # Use default show_cam_on_image for non-cutoff mode
        visualization = show_cam_on_image(image_float, grayscale_cam, use_rgb=True)
    
    # Display results
    num_cols = 3 if show_colorbar else 2
    fig, axes = plt.subplots(1, num_cols, figsize=(5 * num_cols, 5), dpi=300)
    
    # Original image
    axes[0].imshow(image_float)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Overlay
    axes[1].imshow(visualization)
    cutoff_text = f" (cutoff: {cutoff_threshold})" if use_cutoff else ""
    axes[1].set_title(f'GradCAM Overlay{cutoff_text}\nClass: {list(class_map.keys())[list(class_map.values()).index(target_class)]} | Conf: {confidence:.4f}')
    axes[1].axis('off')
    
    # Colorbar/Heatmap scale
    if show_colorbar:
        im = axes[2].imshow(grayscale_cam_display, cmap='jet')
        axes[2].set_title('GradCAM Heatmap')
        axes[2].axis('off')
        plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()
    
    return visualization

# Example usage:
# visualization = load_and_show_gradcam('path/to/image.png', model, device)
# visualization = load_and_show_gradcam('path/to/image.png', model, device, use_cutoff=True, cutoff_threshold=0.4, show_colorbar=True)
# visualization = load_and_show_gradcam('path/to/image.png', model, device, use_cutoff=False, show_colorbar=False)

In [ ]:
from skimage import color

def analyze_inspiration(image_path, rib_model, lung_model, device, rol_cutoff=84.474):
    """
    Analyze chest X-ray for adequate inspiration using rib and lung segmentation.
    
    Args:
        image_path: Path to the chest X-ray image
        rib_model: Trained rib segmentation model
        lung_model: Trained lung segmentation model
        device: torch device (cpu or cuda)
        rol_cutoff: ROL (Rib-Over-Lung) cutoff threshold for classification (default: 84.474)
    
    Returns:
        overlay: Segmentation overlay image
        rol: Calculated ROL value
        classification: "Adequate inspiration" or "Inadequate inspiration"
    """
    
    def preprocess_for_segmentation(img, device):
        """Preprocess image for segmentation models."""
        preprocessing_fn = smp.encoders.get_preprocessing_fn('resnet34', 'imagenet')
        img = preprocessing_fn(img)
        image = img.astype(np.float64)
        image = np.transpose(image, (2, 0, 1))
        img_input = np.expand_dims(image, 0)
        img_input = torch.from_numpy(img_input).float().to(device)
        return img_input
    
    # Load and preprocess image
    SIZE_X = 512
    SIZE_Y = 512
    image = cv2.imread(image_path, 1)  # Read in BGR mode
    image = cv2.resize(image, (SIZE_Y, SIZE_X), interpolation=cv2.INTER_NEAREST)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image / 255
    image = image.astype(np.float64)
    
    # Preprocess for segmentation
    img_input = preprocess_for_segmentation(image.copy(), device)
    
    # Predict with rib model
    with torch.no_grad():
        r_y_pred = rib_model(img_input)
    r_y_pred_np = torch.squeeze(r_y_pred.detach().cpu()).numpy()
    r_y_pred_argmax = np.argmax(r_y_pred_np, axis=0)
    
    # Predict with lung model
    with torch.no_grad():
        l_y_pred = lung_model(img_input)
    l_y_pred_np = torch.squeeze(l_y_pred.detach().cpu()).numpy()
    l_y_pred_argmax = np.argmax(l_y_pred_np, axis=0)
    
    # Create overlay
    # Set lung mask to 1, rib mask to 2
    l_y_pred_argmax[l_y_pred_argmax > 0] = 1
    r_y_pred_argmax[r_y_pred_argmax > 0] = 2
    
    # Combine masks (overlap will be 3)
    r_l_y_pred_argmax = r_y_pred_argmax + l_y_pred_argmax
    
    # Create colored overlay
    overlay = color.label2rgb(
        r_l_y_pred_argmax,
        image,
        colors=[(0, 0, 100), (100, 0, 0), (0, 100, 0)],  # Blue for lung, Red for rib, Green for overlap
        alpha=0.005,
        bg_label=0,
        bg_color=None
    )
    
    # Calculate ROL (Rib-Over-Lung ratio)
    # Union: all rib pixels (2) + overlap pixels (3)
    # Intersection: only overlap pixels (3)
    union = np.count_nonzero(r_l_y_pred_argmax == 2) + np.count_nonzero(r_l_y_pred_argmax == 3)
    intersec = np.count_nonzero(r_l_y_pred_argmax == 3)
    
    if union > 0:
        rol = (intersec / union) * 100
    else:
        rol = 0
    
    # Classify based on cutoff
    if rol > rol_cutoff:
        classification = "Adequate inspiration"
        color_text = "green"
    else:
        classification = "Inadequate inspiration"
        color_text = "red"
    
    # Display results
    fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=300)
    
    # Original image
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Overlay with segmentation
    axes[1].imshow(overlay)
    axes[1].set_title(f'Segmentation Overlay\nROL: {rol:.2f}% (cutoff: {rol_cutoff}%)\n{classification}',
                     color=color_text, fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed results
    print(f"ROL Value: {rol:.2f}%")
    print(f"ROL Cutoff: {rol_cutoff}%")
    print(f"Classification: {classification}")
    print(f"Interpretation: {'✓' if rol > rol_cutoff else '✗'} The rib-over-lung overlap is {'sufficient' if rol > rol_cutoff else 'insufficient'} for adequate inspiration.")
    
    return overlay, rol, classification

# Example usage:
# overlay, rol, classification = analyze_inspiration('path/to/image.png', rib_model, lung_model, device, rol_cutoff=85.698)

### Compare two approce

In [ ]:
def compare_inspiration_approaches(image_path, rib_model, lung_model, classification_model, device, 
                                  rol_cutoff=84.474, gradcam_cutoff=0.3, show_image_name=True, save_image=False, save_folder_name='results'):
    """
    Compare two approaches for assessing chest X-ray inspiration quality:
    1. DeepInspire: Segmentation-based ROL (Rib-Over-Lung) analysis
    2. Classification: Direct classification with GradCAM visualization
    
    Args:
        image_path: Path to the chest X-ray image
        rib_model: Trained rib segmentation model
        lung_model: Trained lung segmentation model
        classification_model: Trained classification model
        device: torch device (cpu or cuda)
        rol_cutoff: ROL threshold for DeepInspire approach (default: 84.474)
        gradcam_cutoff: GradCAM cutoff threshold (default: 0.3)
    
    Returns:
        results: Dictionary containing results from both approaches and timing info
    """
    import time
    
    # Load original image for display
    SIZE_X = 512
    SIZE_Y = 512
    if image_path.endswith('.dcm'):
        import pydicom
        dicom_image = pydicom.dcmread(image_path)
        original_image = dicom_image.pixel_array
    else:
        original_image = cv2.imread(image_path, 1)
    original_image = cv2.resize(original_image, (SIZE_Y, SIZE_X), interpolation=cv2.INTER_NEAREST)
    original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    original_image = original_image / 255
    original_image = original_image.astype(np.float64)
        
    # ============================
    # Approach 1: DeepInspire (ROL-based Segmentation)
    # ============================
    print("=" * 60)
    print("APPROACH 1: DeepInspire (Segmentation-based ROL Analysis)")
    print("=" * 60)
    
    start_time_deepinspire = time.time()
    
    def preprocess_for_segmentation(img, device):
        """Preprocess image for segmentation models."""
        preprocessing_fn = smp.encoders.get_preprocessing_fn('resnet34', 'imagenet')
        img = preprocessing_fn(img)
        image = img.astype(np.float64)
        image = np.transpose(image, (2, 0, 1))
        img_input = np.expand_dims(image, 0)
        img_input = torch.from_numpy(img_input).float().to(device)
        return img_input
    
    # Preprocess for segmentation
    img_input = preprocess_for_segmentation(original_image.copy(), device)
    
    # Predict with rib model
    with torch.no_grad():
        r_y_pred = rib_model(img_input)
    r_y_pred_np = torch.squeeze(r_y_pred.detach().cpu()).numpy()
    r_y_pred_argmax = np.argmax(r_y_pred_np, axis=0)
    
    # Predict with lung model
    with torch.no_grad():
        l_y_pred = lung_model(img_input)
    l_y_pred_np = torch.squeeze(l_y_pred.detach().cpu()).numpy()
    l_y_pred_argmax = np.argmax(l_y_pred_np, axis=0)
    
    # Create overlay
    l_y_pred_argmax[l_y_pred_argmax > 0] = 1
    r_y_pred_argmax[r_y_pred_argmax > 0] = 2
    r_l_y_pred_argmax = r_y_pred_argmax + l_y_pred_argmax
    
    # Create colored overlay
    overlay_deepinspire = color.label2rgb(
        r_l_y_pred_argmax,
        original_image,
        colors=[(0, 0, 100), (100, 0, 0), (0, 100, 0)],
        alpha=0.005,
        bg_label=0,
        bg_color=None
    )
    
    # Calculate ROL
    union = np.count_nonzero(r_l_y_pred_argmax == 2) + np.count_nonzero(r_l_y_pred_argmax == 3)
    intersec = np.count_nonzero(r_l_y_pred_argmax == 3)
    
    if union > 0:
        rol = (intersec / union) * 100
    else:
        rol = 0
    
    # Classify based on ROL
    if rol > rol_cutoff:
        classification_deepinspire = "Adequate inspiration"
    else:
        classification_deepinspire = "Inadequate inspiration"
    
    time_deepinspire = time.time() - start_time_deepinspire
    
    print(f"ROL Value: {rol:.2f}%")
    print(f"ROL Cutoff: {rol_cutoff}%")
    print(f"Classification: {classification_deepinspire}")
    print(f"Processing Time: {time_deepinspire:.3f} seconds")
    
    # ============================
    # Approach 2: Classification with GradCAM
    # ============================
    print("\n" + "=" * 60)
    print("APPROACH 2: Direct Classification with GradCAM")
    print("=" * 60)
    
    start_time_classification = time.time()
    
    # Load and preprocess for classification
    img, image_float = load_image(image_path)
    input_tensor = img.unsqueeze(0).to(device)
    class_map = {"Adequate inspiration": 0, "Inadequate inspiration": 1}
    
    # Get prediction
    with torch.no_grad():
        output = classification_model(input_tensor)
        predicted_class = torch.argmax(output, dim=1).item()
        confidence = torch.softmax(output, dim=1)[0, predicted_class].item()
    
    classification_gradcam = list(class_map.keys())[list(class_map.values()).index(predicted_class)]
    
    # Generate GradCAM
    target_layer = classification_model.layer4[-1]
    cam = GradCAM(model=classification_model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)
    grayscale_cam = grayscale_cam[0, :]
    
    # Apply cutoff
    mask = grayscale_cam < gradcam_cutoff
    grayscale_cam = np.where(mask, 0, grayscale_cam)
    if grayscale_cam.max() > 0:
        grayscale_cam = (grayscale_cam - grayscale_cam.min()) / (grayscale_cam.max() - grayscale_cam.min())
    
    # Create visualization with transparency
    from matplotlib import cm
    colormap = cm.get_cmap('jet')
    heatmap = colormap(grayscale_cam)[:, :, :3]
    alpha = (grayscale_cam > 0).astype(float) * 0.5
    
    visualization_gradcam = image_float.copy()
    for i in range(3):
        visualization_gradcam[:, :, i] = image_float[:, :, i] * (1 - alpha) + heatmap[:, :, i] * alpha
    visualization_gradcam = (visualization_gradcam * 255).astype(np.uint8) / 255.0
    
    time_classification = time.time() - start_time_classification
    
    print(f"Predicted Class: {classification_gradcam}")
    print(f"Confidence: {confidence:.4f}")
    print(f"GradCAM Cutoff: {gradcam_cutoff}")
    print(f"Processing Time: {time_classification:.3f} seconds")
    
    # ============================
    # Display Comparison
    # ============================
    print("\n" + "=" * 60)
    print("COMPARISON SUMMARY")
    print("=" * 60)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=300)
    
    # Original image
    axes[0].imshow(original_image)
    if show_image_name:
        axes[0].set_title(f'Original Image\n({image_path})', fontsize=14, fontweight='bold')
    else:
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # GradCAM visualization
    axes[1].imshow(visualization_gradcam)
    color_gradcam = 'green' if classification_gradcam == "Adequate inspiration" else 'red'
    axes[1].set_title(f'Classification GradCAM\n{classification_gradcam}\nConfidence: {confidence:.2%}\nTime: {time_classification:.3f}s',
                     fontsize=12, fontweight='bold', color=color_gradcam)
    axes[1].axis('off')
    
    # DeepInspire segmentation
    axes[2].imshow(overlay_deepinspire)
    color_deepinspire = 'green' if classification_deepinspire == "Adequate inspiration" else 'red'
    axes[2].set_title(f'DeepInspire \n{classification_deepinspire}\nROL: {rol:.2f}% (cutoff: {rol_cutoff}%)\nTime: {time_deepinspire:.3f}s',
                     fontsize=12, fontweight='bold', color=color_deepinspire)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if save_image:
        os.makedirs(save_folder_name, exist_ok=True)
        base_name = os.path.basename(image_path).replace('.png', '')
        save_path = os.path.join(save_folder_name, f"{base_name}_comparison.png")
        fig.savefig(save_path, bbox_inches='tight')
        print(f"Comparison image saved to: {save_path}")
    
    # Agreement analysis
    agreement = classification_deepinspire == classification_gradcam
    agreement_status = "✓ AGREE" if agreement else "✗ DISAGREE"
    
    print(f"\nDeepInspire Result: {classification_deepinspire} (ROL: {rol:.2f}%)")
    print(f"Classification Result: {classification_gradcam} (Confidence: {confidence:.2%})")
    print(f"Agreement: {agreement_status}")
    print(f"\nTiming Comparison:")
    print(f"  - DeepInspire : {time_deepinspire:.3f} seconds")
    print(f"  - Classification CNN: {time_classification:.3f} seconds")
    faster_approach = "Classification CNN" if time_classification < time_deepinspire else "DeepInspire"
    time_diff = abs(time_classification - time_deepinspire)
    print(f"  - {faster_approach} is faster by {time_diff:.3f} seconds")
    
    # Return results dictionary
    results = {
        'deepinspire': {
            'classification': classification_deepinspire,
            'rol': rol,
            'time': time_deepinspire,
            'overlay': overlay_deepinspire
        },
        'classification': {
            'classification': classification_gradcam,
            'confidence': confidence,
            'time': time_classification,
            'visualization': visualization_gradcam
        },
        'agreement': agreement,
        'original_image': original_image
    }
    
    return results

# Example usage:
# results = compare_inspiration_approaches('image/full/151.png', rib_model, lung_model, model, device)

## Adequate inspiration

In [ ]:
import os
full_image_path = os.listdir('../2026 Dataset/test/full/image')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../2026 Dataset/test/full/image', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            save_image=True,
            save_folder_name='comparison_results_full'
        )

## Inadequate inspiration

In [ ]:
import os
full_image_path = os.listdir('../2026 Dataset/test/not full/image')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../2026 Dataset/test/not full/image', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            save_image=True,
            save_folder_name='comparison_results_notfull'
        )

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Artifact')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Artifact', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4
        )

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Only DeepInspire Correct/GT full')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Only DeepInspire Correct/GT full', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False
        )

## Only DeepInspire correct

### GT full

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Only DeepInspire Correct/GT full')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Only DeepInspire Correct/GT full', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='OnlyDeepInspire_correct_GTfull'
        )

### GT not full

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Only DeepInspire Correct/GT not full')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Only DeepInspire Correct/GT not full', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='OnlyDeepInspire_correct_GTnotfull'
        )

## Only CNN correct

### GT full (0 images)

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Only CNN Correct/GT full')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Only CNN Correct/GT full', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False
        )

### GT not full

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Only CNN Correct/GT not full')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Only CNN Correct/GT not full', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='OnlyCNN_correct_GTnotfull'
        )

## DeepInspire False

### False Positive (GT full but predicted as not full)

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/False Image/DeepInspire/FP')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/False Image/DeepInspire/FP', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='DeepInspire_False_GTfull'
        )

### False Negative (GT not full but predicted as full)

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/False Image/DeepInspire/FN')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/False Image/DeepInspire/FN', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='DeepInspire_False_GTnotfull'
        )

## CNN False

### False Negative (GT not full but predicted as full)

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/False Image/CNN/FN')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/False Image/CNN/FN', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='CNN_False_GTnotfull'
        )

### False Positive (GT full but predicted as not full)

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/False Image/CNN/FP')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/False Image/CNN/FP', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='CNN_False_GTfull'
        )

## Both False

### GT full but predicted as not full

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Both wrong/GT full')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Both wrong/GT full', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='Both_False_GTnotfull'
        )

### GT not full but predicted as full

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Both wrong/GT not full')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Both wrong/GT not full', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='Both_False_GTfull'
        )

## Patho

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Patho')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Patho', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='Our_Patho'
        )

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Interner_Patho')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Interner_Patho', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.4,
            show_image_name=False,
            save_image=True,
            save_folder_name='Internet_Patho'
        )

## Out

In [ ]:
import os
folder_image_path = os.listdir('../2026 Dataset/out/image')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../2026 Dataset/out/image', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.2,
            show_image_name=False
        )

## Artifact

In [ ]:
import os
folder_image_path = os.listdir('../Focus image/Artifact')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Artifact', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Artifact'
        )

In [ ]:
import os
folder_image_path = os.listdir('../Neo WyvernGUI/!Test img/H_Artifact')
for img_name in folder_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Neo WyvernGUI/!Test img/H_Artifact', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False
        )

## Crop FP

In [ ]:
image_path = '../test_full_193.png'
results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False
        )

## Plural Efffusion

In [ ]:
image_path = '../pf3.jpeg'
results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False
        )

## Pace maker

In [ ]:
import os
full_image_path = os.listdir('../Focus image/Pace maker')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Focus image/Pace maker', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            save_image=True,
            save_folder_name='pace_maker_results'
        )

## Public Dataset Pathology (CASIA-CXR)

Link to the dataset: https://www.casia-cxr.net/index.html

### Cardiomegaly

In [ ]:
import os
full_image_path = os.listdir('../CASIA-CXR_Sample/CASIA-CXR_Cardiomegaly_Sample/CASIA-CXR_Cardiomegaly_Images')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../CASIA-CXR_Sample/CASIA-CXR_Cardiomegaly_Sample/CASIA-CXR_Cardiomegaly_Images', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='cardiomegaly_results'
        )

### Mass

In [ ]:
import os
full_image_path = os.listdir('../CASIA-CXR_Sample/CASIA-CXR_Mass_Sample/CASIA-CXR_Mass_Images')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../CASIA-CXR_Sample/CASIA-CXR_Mass_Sample/CASIA-CXR_Mass_Images', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='mass_results'
        )

### Pleural Effusion

In [ ]:
import os
full_image_path = os.listdir('../CASIA-CXR_Sample/CASIA-CXR_PleuralEffusion_Sample/CASIA-CXR_PleuralEffusion_Images')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../CASIA-CXR_Sample/CASIA-CXR_PleuralEffusion_Sample/CASIA-CXR_PleuralEffusion_Images', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='pleural_effusion_results'
        )

### Pneumonia

In [ ]:
import os
full_image_path = os.listdir('../CASIA-CXR_Sample/CASIA-CXR_Pneumonia_Sample/CASIA-CXR_Pneumonia_Images')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../CASIA-CXR_Sample/CASIA-CXR_Pneumonia_Sample/CASIA-CXR_Pneumonia_Images', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='pneumonia_results'
        )

### Penumothorax

In [ ]:
import os
full_image_path = os.listdir('../CASIA-CXR_Sample/CASIA-CXR_Pneumothorax_Sample/CASIA-CXR_Pneumothorax_Images')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../CASIA-CXR_Sample/CASIA-CXR_Pneumothorax_Sample/CASIA-CXR_Pneumothorax_Images', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='pneumothorax_results'
        )

## Public Dataset Pathology (Chest X-ray DICOM)

Link to the dataset: [https://www.casia-cxr.net/index.html ](https://www.kaggle.com/datasets/falahgatea/chest-x-ray-dicom)

### Pnumothorax

In [ ]:
import os
full_image_path = os.listdir('../Chest-X-Ray-DICOM/Pneumothorax')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Chest-X-Ray-DICOM/Pneumothorax', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-DICOM-pneumothorax_results'
        )

### No Pneumothorax

In [ ]:
import os
full_image_path = os.listdir('../Chest-X-Ray-DICOM/No Pneumothorax')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../Chest-X-Ray-DICOM/No Pneumothorax', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-DICOM-no-pneumothorax_results'
        )

## Public Dataset Pathology (Chest X-ray 13)

Link to the dataset: [https://www.kaggle.com/datasets/falahgatea/chest-x-ray-dicom](https://www.kaggle.com/datasets/animeshshedge/chest-x-rays-of-14-common-disease)

###  Atelectasis

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Atelectasis')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Atelectasis', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-atelectasis_results'
        )

### Cardiomegaly

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Cardiomegaly')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Cardiomegaly', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-cardiomegaly_results'
        )

### Consolidation

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Consolidation')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Consolidation', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-consolidation_results'
        )

### Edema

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Edema')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Edema', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-edema_results'
        )

### Effusion

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Effusion')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Effusion', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-effusion_results'
        )

### Emphysema

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Emphysema')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Emphysema', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-emphysema_results'
        )

### Fibrosis

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Fibrosis')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Fibrosis', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-fibrosis_results'
        )

### Infiltration

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Inflitration')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Inflitration', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-infiltration_results'
        )

### Mass

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Mass')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Mass', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-mass_results'
        )

### Nodule

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Nodule')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Nodule', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-nodule_results'
        )

### Pleural Thickening

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Pleural_Thickening')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Pleural_Thickening', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-pleural_thickening_results'
        )

### Pneumonia

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Pneumonia')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Pneumonia', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-pneumonia_results'
        )

### Pneumothorax

In [ ]:
import os
full_image_path = os.listdir('../ChestX-ray13/Pneumothorax')
for img_name in full_image_path:
    print(f"\nProcessing image: {img_name}")
    image_path = os.path.join('../ChestX-ray13/Pneumothorax', img_name)
    if '.DS_Store' in image_path:
        continue
    else:
        results = compare_inspiration_approaches(
            image_path, 
            rib_model,
            lung_model, 
            model, 
            device, 
            rol_cutoff=83.92, 
            gradcam_cutoff=0.5,
            show_image_name=False,
            save_image=True,
            save_folder_name='Chest-X-Ray-13-pneumothorax_results'
        )